## Read MLS raw data
Raw MLS data contains formatting inconsistencies, missing values, and fields that need transformation
before analysis. This phase prepares the dataset for reliable analytics.

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import csv

In [16]:
# Load the dataset
listing = pd.read_csv('listing.csv')

/var/folders/fb/y3cbnxzj3rs43kpb74nk2f0r0000gn/T/ipykernel_17351/4100435822.py:2: DtypeWarning: Columns (2,43) have mixed types. Specify dtype option on import or set low_memory=False.
  listing = pd.read_csv('listing.csv')


In [17]:
listing.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency',
       'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor',
       'TaxAnnualAmount', 'CountyOrParish', 'PropertyType.1', 'MlsStatus',
       'ElementarySchool', 'ListAgentFirstName.1', 'AttachedGarageYN',
       'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'DaysOnMarket.1',
       'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'LivingArea.1',
       'ListingId', 'BathroomsTotalInteger', 'Cit

In [18]:
listing_cleaned = listing.copy()
listing_cleaned.head()

,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1
0,90000.0,1075010398,miriamlara03@gmail.com,NaN,NaN,Miriam,Lara,34.097939,-117.909653,1045 N Azusa 61,...,NaN,NaN,2.0,Covina Valley Unified,91722,NaN,0.0,NaN,NaN,1045 N Azusa 61
1,1500000.0,1074974457,janelle@judsonre.com,NaN,NaN,Janelle,Judson,33.121241,-117.081614,NaN,...,NaN,NaN,NaN,NaN,92025,NaN,0.0,NaN,NaN,NaN
2,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,NaN,False,NaN,NaN,90067,NaN,2105.0,177861.0,NaN,2220 Avenue Of The Stars 2704
3,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,0.0,False,3.0,Capistrano Unified,92677,NaN,254.0,5300.0,NaN,16 Palisades
4,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,NaN,NaN,2.0,NaN,91108,NaN,NaN,9404.0,NaN,1615 Waverly Road


### Convert date fields to datetime format

In [19]:
date_columns = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']
for col in date_columns:
    listing_cleaned[col] = pd.to_datetime(listing[col], errors='coerce')

In [20]:
# listing_cleaned.info()

### Remove unnecessary or redundant columns
Here we only retain core analysis columns with low missing ratio 

In [21]:
listing_retain = [
    'OriginalListPrice', 'ListPrice', 'ClosePrice',
    'CloseDate', 'ListingContractDate', 'PurchaseContractDate',
    'DaysOnMarket','ContractStatusChangeDate', 'LivingArea', 'LotSizeAcres',
    'BedroomsTotal', 'BathroomsTotalInteger', 'YearBuilt',
    'City', 'PostalCode', 'CountyOrParish',
    'Latitude', 'Longitude',
    'PropertyType', 'PropertySubType', 'MlsStatus', 'StateOrProvince'
]

listing_cleaned = listing_cleaned[listing_retain]
listing_cleaned.head()

,OriginalListPrice,ListPrice,ClosePrice,CloseDate,ListingContractDate,PurchaseContractDate,DaysOnMarket,ContractStatusChangeDate,LivingArea,LotSizeAcres,...,YearBuilt,City,PostalCode,CountyOrParish,Latitude,Longitude,PropertyType,PropertySubType,MlsStatus,StateOrProvince
0,90000.0,90000.0,NaN,NaT,2024-01-29,NaT,0,2024-01-29,960.0,NaN,...,1960.0,Covina,91722,Los Angeles,34.097939,-117.909653,ManufacturedInPark,NaN,Active,CA
1,1500000.0,1500000.0,NaN,NaT,2024-01-11,NaT,0,2024-01-11,NaN,NaN,...,1940.0,Escondido,92025,San Diego,33.121241,-117.081614,CommercialSale,Retail,Active,CA
2,1340000.0,1340000.0,NaN,NaT,2024-01-01,2024-05-07,127,2024-05-07,1301.0,4.0831,...,1964.0,Los Angeles,90067,Los Angeles,34.052207,-118.408445,Residential,Condominium,Pending,CA
3,2500000.0,2500000.0,NaN,NaT,2024-01-24,NaT,1,2024-01-24,2788.0,0.1217,...,1994.0,Laguna Niguel,92677,Orange,33.496363,-117.691677,Residential,SingleFamilyResidence,Active,CA
4,3150000.0,3150000.0,NaN,NaT,2024-01-12,NaT,1,2024-01-12,3250.0,0.2159,...,1965.0,San Marino,91108,Los Angeles,34.119345,-118.111254,Residential,SingleFamilyResidence,Active,CA


### Missing Value Treatment

In [22]:
# highmissing threshold
p_high = 0.5
p_very_high = 0.9

listing_missing_summary = pd.DataFrame({
    'missing_count': listing_cleaned.isna().sum(),
    'missing_ratio': listing_cleaned.isna().mean()
}).sort_values('missing_ratio', ascending=False)

listing_high_missing_summary = listing_missing_summary[listing_missing_summary['missing_ratio'] > p_high].copy()

listing_high_missing_summary['extreme_high_missing_flag'] = (listing_high_missing_summary['missing_ratio'] > p_very_high).astype(int)

listing_high_missing_summary

,missing_count,missing_ratio,extreme_high_missing_flag
ClosePrice,613920,0.719750,0
CloseDate,591323,0.693258,0
PurchaseContractDate,479201,0.561807,0


Based on the result, we can see that we've ingored the extreme high missing columns, for those still in high missing value we can either leave, label or drop them depends on what analysis we want to do.

Fill NaN for missing value

In [23]:
listing_cleaned.fillna(np.nan, inplace=True)


### Ensure numeric fields are properly typed

In [24]:
listing_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 852963 entries, 0 to 852962
Data columns (total 22 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   OriginalListPrice         849641 non-null  float64       
 1   ListPrice                 850826 non-null  float64       
 2   ClosePrice                239043 non-null  float64       
 3   CloseDate                 261640 non-null  datetime64[ns]
 4   ListingContractDate       852963 non-null  datetime64[ns]
 5   PurchaseContractDate      373762 non-null  datetime64[ns]
 6   DaysOnMarket              852963 non-null  int64         
 7   ContractStatusChangeDate  846123 non-null  datetime64[ns]
 8   LivingArea                746485 non-null  float64       
 9   LotSizeAcres              771538 non-null  float64       
 10  BedroomsTotal             749881 non-null  float64       
 11  BathroomsTotalInteger     782985 non-null  float64       
 12  Ye

In [25]:
pd.set_option('display.max_columns', None)
listing_cleaned.head()

,OriginalListPrice,ListPrice,ClosePrice,CloseDate,ListingContractDate,PurchaseContractDate,DaysOnMarket,ContractStatusChangeDate,LivingArea,LotSizeAcres,BedroomsTotal,BathroomsTotalInteger,YearBuilt,City,PostalCode,CountyOrParish,Latitude,Longitude,PropertyType,PropertySubType,MlsStatus,StateOrProvince
0,90000.0,90000.0,NaN,NaT,2024-01-29,NaT,0,2024-01-29,960.0,NaN,4.0,1.0,1960.0,Covina,91722,Los Angeles,34.097939,-117.909653,ManufacturedInPark,NaN,Active,CA
1,1500000.0,1500000.0,NaN,NaT,2024-01-11,NaT,0,2024-01-11,NaN,NaN,NaN,0.0,1940.0,Escondido,92025,San Diego,33.121241,-117.081614,CommercialSale,Retail,Active,CA
2,1340000.0,1340000.0,NaN,NaT,2024-01-01,2024-05-07,127,2024-05-07,1301.0,4.0831,2.0,2.0,1964.0,Los Angeles,90067,Los Angeles,34.052207,-118.408445,Residential,Condominium,Pending,CA
3,2500000.0,2500000.0,NaN,NaT,2024-01-24,NaT,1,2024-01-24,2788.0,0.1217,4.0,3.0,1994.0,Laguna Niguel,92677,Orange,33.496363,-117.691677,Residential,SingleFamilyResidence,Active,CA
4,3150000.0,3150000.0,NaN,NaT,2024-01-12,NaT,1,2024-01-12,3250.0,0.2159,6.0,4.0,1965.0,San Marino,91108,Los Angeles,34.119345,-118.111254,Residential,SingleFamilyResidence,Active,CA


#### Change BatheroomsTotalInteger and YearBuilt Column to int and keep missing values at the same time

In [26]:
listing_cleaned['BathroomsTotalInteger'] = listing_cleaned['BathroomsTotalInteger'].astype('Int64')
listing_cleaned['YearBuilt'] = listing_cleaned['YearBuilt'].astype('Int64')

### Remove or flag invalid numeric values

In [27]:
# filter with numeric columns and check for negative values
numeric_cols = listing_cleaned.select_dtypes(include=['int64', 'float64', 'Int64']).columns

negative_summary = []

for col in numeric_cols:
    negative_count = (listing_cleaned[col] < 0).sum()
    
    # Only include columns with negative values in the summary
    if negative_count > 0:
        negative_summary.append({
            'column': col,
            'negative_count': negative_count,
            'min_value': listing_cleaned[col].min()
        })

negative_summary = pd.DataFrame(negative_summary)

negative_summary

,column,negative_count,min_value
0,DaysOnMarket,37,-58.000000
1,Latitude,7,-117.472493
2,Longitude,741500,-159.523315


Latitude and Longitude can be negative so invalid values only happen in DaysOnMarket

In [28]:
# drop rows with negative values in 'DaysOnMarket'
listing_cleaned = listing_cleaned[listing_cleaned['DaysOnMarket'] >= 0].copy()
(listing_cleaned['DaysOnMarket'] < 0).sum()

np.int64(0)

## Data Consistency Check 
Validate the logical order of date fields: ListingContractDate should precede PurchaseContractDate,
which should precede CloseDate. Create boolean flag columns to mark records that violate these rules:
-  listing_after_close_flag
- purchase_after_close_flag
- negative_timeline_flag

In [30]:
listing_cleaned['listing_after_purchase_flag'] = (listing_cleaned['ListingContractDate'] > listing_cleaned['PurchaseContractDate'])
listing_cleaned['listing_after_purchase_flag'].value_counts()
listing_cleaned['purchase_after_close_flag'] = (listing_cleaned['PurchaseContractDate'] > listing_cleaned['CloseDate'])
listing_cleaned['purchase_after_close_flag'].value_counts()
listing_cleaned['negative_timeline_flag'] = listing_cleaned['listing_after_purchase_flag'] | listing_cleaned['purchase_after_close_flag']
listing_cleaned['negative_timeline_flag'].value_counts()

negative_timeline_flag
False    852179
True        747
Name: count, dtype: int64

## Geographic Data Checks
- Flag records with missing coordinates (Latitude or Longitude is null)
- Flag Latitude = 0 or Longitude = 0 (sentinel null values)
- Flag Longitude > 0 errors (California coordinates should be negative)
- Flag out-of-state or implausible coordinates

In [31]:
# either latitude or longitude is missing
listing_cleaned['missing_coordinates_flag'] = listing_cleaned['Latitude'].isna() | listing_cleaned['Longitude'].isna()
listing_cleaned['missing_coordinates_flag'].value_counts()

missing_coordinates_flag
False    741014
True     111912
Name: count, dtype: int64

In [32]:
# either latitude or longitude == 0
listing_cleaned['zero_coordinates_flag'] = (listing_cleaned['Latitude'] == 0) | (listing_cleaned['Longitude'] == 0)
listing_cleaned['zero_coordinates_flag'].value_counts()

zero_coordinates_flag
False    852832
True         94
Name: count, dtype: int64

In [33]:
# longitude > 0 (should be negative in the CA)
listing_cleaned['invalid_longitude_flag'] = listing_cleaned['Longitude'] > 0
listing_cleaned['invalid_longitude_flag'].value_counts()

invalid_longitude_flag
False    852763
True        163
Name: count, dtype: int64

Realize that some transactions are from other State and should be deal with

In [34]:
listing_cleaned['StateOrProvince'].value_counts()

StateOrProvince
CA    852555
OS       117
AZ        26
NV        16
TX        11
FL         9
AL         9
OR         9
WA         9
OK         8
BC         7
HI         7
NM         6
MO         6
CO         6
ID         5
NY         4
OH         4
SC         4
TN         3
MT         3
ME         3
VT         2
NJ         2
AR         2
VA         2
LA         2
MI         2
MS         1
AK         1
MN         1
DC         1
PA         1
UT         1
GA         1
IL         1
IA         1
MD         1
NC         1
KS         1
Name: count, dtype: int64

In [35]:
listing_cleaned['implausible_coordinate_flag'] = (
    (listing_cleaned['Latitude'] < 32) |
    (listing_cleaned['Latitude'] > 42) |
    (listing_cleaned['Longitude'] < -125) |
    (listing_cleaned['Longitude'] > -114)
)
listing_cleaned['implausible_coordinate_flag'].value_counts()

implausible_coordinate_flag
False    852403
True        523
Name: count, dtype: int64

## Output a cleaned data

In [36]:
listing_cleaned.to_csv('listing_cleaned.csv', index=False) 

In [37]:
listing_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 852926 entries, 0 to 852962
Data columns (total 29 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   OriginalListPrice            849604 non-null  float64       
 1   ListPrice                    850789 non-null  float64       
 2   ClosePrice                   239020 non-null  float64       
 3   CloseDate                    261617 non-null  datetime64[ns]
 4   ListingContractDate          852926 non-null  datetime64[ns]
 5   PurchaseContractDate         373734 non-null  datetime64[ns]
 6   DaysOnMarket                 852926 non-null  int64         
 7   ContractStatusChangeDate     846086 non-null  datetime64[ns]
 8   LivingArea                   746450 non-null  float64       
 9   LotSizeAcres                 771501 non-null  float64       
 10  BedroomsTotal                749846 non-null  float64       
 11  BathroomsTotalInteger        78